# Gold · Hechos de Detalle de Pedido

## Propósito

Construir la tabla de hechos de detalle de pedido en la capa Gold, con una línea por pedido y producto.

## Flujo

1. Leer el detalle de pedido desde la capa Silver.
2. Relacionar cada línea con el producto y con la versión vigente del pedido (`es_vigente = true`), no con versiones cerradas por cambio de estado.
3. Escribir la tabla de hechos.

## Destino

Capa Gold.

**Tabla:** `gld_dataco.supply_chain.desc_ft_order_details`

**Llaves:** `orden_key` y `producto_key`

**Grano:** una línea de detalle por pedido y producto

## Métricas que aporta

* Ventas
* Utilidad
* Cantidad
* Beneficio por línea

## Resultado esperado

El detalle de pedido relacionado con la versión vigente de cada pedido, listo para que el resumen por pedido lo agregue.

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 1: Parámetros
# ============================================================

dbutils.widgets.text("silver_catalog", "slv_dataco", "Catálogo Silver")
dbutils.widgets.text("silver_schema", "supply_chain", "Esquema Silver")

dbutils.widgets.text("gold_catalog", "gld_dataco", "Catálogo Gold")
dbutils.widgets.text("gold_schema", "supply_chain", "Esquema Gold")

silver_catalog = dbutils.widgets.get("silver_catalog")
silver_schema = dbutils.widgets.get("silver_schema")
gold_catalog = dbutils.widgets.get("gold_catalog")
gold_schema = dbutils.widgets.get("gold_schema")

SILVER_PEDIDO_DETALLE_FQN = f"{silver_catalog}.{silver_schema}.hd_pedido_detalle"
GOLD_DIM_PRODUCTS_FQN = f"{gold_catalog}.{gold_schema}.desc_dim_products"
GOLD_FACT_ORDERS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_orders"
GOLD_FACT_ORDER_DETAILS_FQN = f"{gold_catalog}.{gold_schema}.desc_ft_order_details"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_catalog}.{gold_schema}")

print(f"✓ Origen: {SILVER_PEDIDO_DETALLE_FQN}")
print(f"✓ Destino: {GOLD_FACT_ORDER_DETAILS_FQN}")

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 2: INICIO DE LOG DE EJECUCIÓN
# ============================================================

import uuid

id_ejecucion = dbutils.jobs.taskValues.get(
    taskKey="prepare_pipeline_control",
    key="id_ejecucion",
    debugValue=""
)

LOG_ID = str(uuid.uuid4())
NOTEBOOK = "gld_fact_order_details"
CAPA = "gold"

spark.sql(f"""
INSERT INTO metadata_dataco.supply_chain.hd_etl_log
(log_id, id_ejecucion, notebook, capa, estado, fecha_inicio)
VALUES ('{LOG_ID}', '{id_ejecucion}', '{NOTEBOOK}', '{CAPA}', 'EN_PROCESO', current_timestamp())
""")

dbutils.jobs.taskValues.set(key="log_id", value=LOG_ID)

print(f"✓ Registro EN_PROCESO creado en etl_log: {LOG_ID}")

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 3: Leer Silver + dimensiones
# ------------------------------------------------------------
# fact_orders se filtra por es_vigente = true: cada línea de
# detalle se relaciona con la versión ACTUAL del pedido, no con
# versiones cerradas por cambio de estado.
# ============================================================

from pyspark.sql import functions as F

df_detalle = spark.table(SILVER_PEDIDO_DETALLE_FQN)

df_dim_products = (
    spark.table(GOLD_DIM_PRODUCTS_FQN)
    .filter(F.col("es_vigente") == True)
    .select("product_card_id", "producto_key")
)

df_fact_orders_vigente = (
    spark.table(GOLD_FACT_ORDERS_FQN)
    .filter(F.col("es_vigente") == True)
    .select("order_id", "orden_key")
)

print(f"✓ Líneas de detalle en Silver: {df_detalle.count()}")

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 4: Join contra dimensiones y hecho de pedidos
# ============================================================

df_fact_order_details = (
    df_detalle.alias("d")
    .join(df_dim_products.alias("p"), F.col("d.order_item_cardprod_id") == F.col("p.product_card_id"), "left")
    .join(df_fact_orders_vigente.alias("o"), F.col("d.order_id") == F.col("o.order_id"), "left")
    .select(
        F.col("d.order_item_id"),
        F.col("o.orden_key"),
        F.col("p.producto_key"),
        F.col("d.order_item_quantity"),
        F.col("d.order_item_product_price"),
        F.col("d.order_item_discount"),
        F.col("d.order_item_discount_rate"),
        F.col("d.order_item_profit_ratio"),
        F.col("d.sales"),
        F.col("d.order_item_total"),
        F.col("d.order_profit_per_order"),
        F.col("d.benefit_per_order"),
        F.col("d.sales_per_customer"),
        F.col("d.fecha_inicio_corte"),
        F.col("d.fecha_fin_corte"),
        F.col("d.fecha_carga"),
        F.col("d.id_ejecucion")
    )
)

nulos_producto = df_fact_order_details.filter(F.col("producto_key").isNull()).count()
nulos_orden = df_fact_order_details.filter(F.col("orden_key").isNull()).count()

print(f"✓ Filas en fact_order_details: {df_fact_order_details.count()}")
print(f"⚠️ Sin match en dim_products: {nulos_producto}")
print(f"⚠️ Sin match en fact_orders (pedido no vigente o cancelado): {nulos_orden}")

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 5: Escritura
# ============================================================

(
    df_fact_order_details.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(GOLD_FACT_ORDER_DETAILS_FQN)
)

print(f"✓ Tabla {GOLD_FACT_ORDER_DETAILS_FQN} escrita: {df_fact_order_details.count()} filas")

In [0]:
# ============================================================
# GLD_FACT_ORDER_DETAILS — Celda 6: ACTUALIZACIÓN DEL LOG — EJECUCIÓN EXITOSA
# ============================================================

spark.sql(f"""
UPDATE metadata_dataco.supply_chain.hd_etl_log
SET estado = 'EXITOSO', fecha_fin = current_timestamp()
WHERE log_id = '{LOG_ID}'
""")

print("✓ etl_log actualizado: EXITOSO")